## 0. Imports & setup

Standard data science libraries plus `requests` for TMDB API calls and `time` for rate-limit sleep.
`DATA_DIR` points to the folder where raw MovieLens files are stored and where all outputs will be saved.

In [1]:
import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path

DATA_DIR = Path('../data')
TMDB_API_KEY = '4ef7fb9aeb9935e011c8af3c495cdd6c'

## 1. Load MovieLens data

### Why two data sources?

**MovieLens 25M** gives us what TMDB cannot: **25 million real user ratings** across 60k+ movies. This is the foundation for Option 2 (personalised recommendations via SVD) and also lets us rank movies by popularity to decide which ones are worth fetching.

**TMDB API** gives us what MovieLens cannot: **rich movie metadata** — text descriptions (overviews), release dates, directors, budgets. MovieLens only has title and pipe-separated genres, which is not enough to build meaningful content embeddings.

Together they complement each other: MovieLens for behaviour signals, TMDB for content signals.

---

MovieLens 25M contains three files we care about:
- **movies.csv** — movieId, title, pipe-separated genres
- **ratings.csv** — userId, movieId, rating (0.5–5.0), timestamp
- **links.csv** — movieId → tmdbId mapping needed to call the TMDB API

We load all three separately and merge them in the next steps.

In [2]:
# Load MovieLens movies
movies = pd.read_csv(DATA_DIR / 'movies.csv')
print(movies.shape)
movies.head()

(62423, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
# Load ratings
ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
print(ratings.shape)
ratings.head()

(25000095, 4)


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [4]:
# links.csv contains tmdbId for each movieId
links = pd.read_csv(DATA_DIR / 'links.csv')
print(links.shape)
links.head()

(62423, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [5]:
# Merge movies with tmdbId
movies = movies.merge(links[['movieId', 'tmdbId']], on='movieId', how='left')
movies['tmdbId'] = movies['tmdbId'].dropna().astype(int)
print(f'Movies with tmdbId: {movies["tmdbId"].notna().sum()} / {len(movies)}')
movies.head()

Movies with tmdbId: 62316 / 62423


,movieId,title,genres,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0
4,5,Father of the Bride Part II (1995),Comedy,11862.0


## 2. TMDB API — fetch all available fields in one request

`get_movie_details` uses `append_to_response=credits,keywords` to retrieve everything TMDB offers in a **single HTTP request** per movie. Fields will be filtered in the next step — here we store everything raw.

| Group | Fields |
|---|---|
| Basic info | `overview`, `tagline`, `original_title`, `original_language`, `release_date`, `release_year`, `runtime`, `status` |
| Scores | `vote_average`, `vote_count`, `popularity` |
| Money | `budget`, `revenue` |
| Images | `poster_path`, `backdrop_path` |
| Geography | `origin_country`, `country`, `spoken_languages` |
| Franchise | `collection` |
| Companies | `production_companies` |
| Credits | `director`, `writer`, `composer`, `cinematographer`, `producer`, `cast` (top 10) |
| Keywords | `keywords` (pipe-separated thematic tags) |

In [6]:
API_KEY = '4ef7fb9aeb9935e011c8af3c495cdd6c'

def pipe(lst):
    """Join a list of strings into a pipe-separated string."""
    return '|'.join(str(x) for x in lst) if lst else None

def get_movie_details(tmdb_id):
    # Fetch everything TMDB offers in one request.
    # append_to_response=credits,keywords adds cast/crew and thematic tags
    # without an extra round-trip.
    # We store all fields raw here — filtering happens in a later step.
    url = f'https://api.themoviedb.org/3/movie/{int(tmdb_id)}'
    r = requests.get(url, params={
        'api_key': API_KEY,
        'append_to_response': 'credits,keywords'
    })
    if r.status_code != 200:
        return None
    d = r.json()

    # ── credits ───────────────────────────────────────────────────────────────
    crew = d.get('credits', {}).get('crew', [])
    cast = d.get('credits', {}).get('cast', [])

    directors    = [p['name'] for p in crew if p['job'] == 'Director']
    writers      = [p['name'] for p in crew if p['job'] in ('Writer', 'Screenplay', 'Story')]
    composers    = [p['name'] for p in crew if p['job'] == 'Original Music Composer']
    cinematogs   = [p['name'] for p in crew if p['job'] == 'Director of Photography']
    producers    = [p['name'] for p in crew if p['job'] == 'Producer']

    # ── simple scalar fields ──────────────────────────────────────────────────
    budget  = d.get('budget',  0)
    revenue = d.get('revenue', 0)

    return {
        # basic info
        'overview':          d.get('overview', ''),
        'tagline':           d.get('tagline', ''),
        'original_title':    d.get('original_title'),
        'original_language': d.get('original_language'),
        'release_date':      d.get('release_date'),
        'release_year':      d.get('release_date', '')[:4],
        'runtime':           d.get('runtime'),
        'status':            d.get('status'),

        # scores
        'vote_average': d.get('vote_average'),
        'vote_count':   d.get('vote_count'),
        'popularity':   d.get('popularity'),

        # money
        'budget':  budget  if budget  and budget  > 0 else None,
        'revenue': revenue if revenue and revenue > 0 else None,

        # images
        'poster_path':   d.get('poster_path'),
        'backdrop_path': d.get('backdrop_path'),

        # geography
        'origin_country':    pipe(d.get('origin_country', [])),
        'country':           pipe([c['iso_3166_1'] for c in d.get('production_countries', [])]),
        'spoken_languages':  pipe([l['english_name'] for l in d.get('spoken_languages', [])]),

        # franchise
        'collection': d.get('belongs_to_collection', {}).get('name') if d.get('belongs_to_collection') else None,

        # companies
        'production_companies': pipe([c['name'] for c in d.get('production_companies', [])]),

        # credits
        'director':       directors[0] if directors else None,
        'writer':         pipe(writers[:3]),
        'composer':       composers[0] if composers else None,
        'cinematographer': cinematogs[0] if cinematogs else None,
        'producer':       pipe(producers[:3]),
        'cast':           pipe([p['name'] for p in cast[:10]]),

        # keywords
        'keywords': pipe([k['name'] for k in d.get('keywords', {}).get('keywords', [])]),
    }


## 3. Select top 10k most-rated movies

Fetching all 60k+ movies in MovieLens would take several hours.
We limit to the **10,000 most-rated** movies — these are the most popular and are most likely to have complete TMDB data (overview, budget, director).
`n_ratings` is used as a proxy for popularity within the MovieLens dataset.

In [7]:
# Keep only the most-rated movies to limit API calls.
# rating_counts gives us a proxy for popularity within MovieLens.
rating_counts = ratings.groupby('movieId').size().rename('n_ratings')
movies_popular = (
    movies
    .join(rating_counts, on='movieId')
    .sort_values('n_ratings', ascending=False)
    #.head(30_000)
    .reset_index(drop=True)
)
print(f'Fetching details for {len(movies_popular)} movies')


Fetching details for 62423 movies


## 4. Fetch TMDB details (~7–10 min)

One request per movie. `time.sleep(0.03)` adds a 30ms pause between requests to stay under TMDB's rate limit (~40 req/s).
Progress is printed every 500 movies. Errors (HTTP non-200 or exceptions) are counted but silently skipped — a small number of missing movies is acceptable.

In [8]:
# Single pass over all movies: each request returns overview, release_year,
# vote_average, popularity, budget, director, country, and poster_path in one round-trip.
# sleep(0.03) keeps us under TMDB's ~40 req/s rate limit.
# Every 500 rows we checkpoint to data/tmdb_cache.csv so a crash doesn't lose progress.
# On restart: already-fetched movieIds are skipped automatically.

import os

CACHE_PATH = DATA_DIR / 'tmdb_cache.csv'

# Load existing cache if available — allows resuming after a crash
if CACHE_PATH.exists():
    cached = pd.read_csv(CACHE_PATH)
    results = cached.to_dict('records')
    fetched_ids = set(cached['movieId'])
    print(f'Resuming from cache: {len(results)} already fetched')
else:
    results = []
    fetched_ids = set()

errors = 0

for i, (_, row) in enumerate(movies_popular.iterrows()):
    if row['movieId'] in fetched_ids:
        continue  # already fetched in a previous run

    if i % 500 == 0:
        print(f'{i}/{len(movies_popular)}  fetched={len(results)}  errors={errors}')

    try:
        details = get_movie_details(row['tmdbId'])
        if details:
            results.append({'movieId': row['movieId'], **details})
        time.sleep(0.03)
    except Exception as e:
        errors += 1

    # Checkpoint every 500 new results
    if len(results) % 500 == 0 and len(results) > 0:
        pd.DataFrame(results).to_csv(CACHE_PATH, index=False)

# Final save
tmdb_df = pd.DataFrame(results)
tmdb_df.to_csv(CACHE_PATH, index=False)
print(f'Done: {len(results)} fetched, {errors} errors')


Resuming from cache: 52500 already fetched
23500/62423  fetched=52501  errors=26
27500/62423  fetched=52501  errors=30
53500/62423  fetched=52687  errors=81
54000/62423  fetched=53182  errors=85
54500/62423  fetched=53657  errors=102
55000/62423  fetched=54151  errors=102
55500/62423  fetched=54646  errors=102
56000/62423  fetched=55125  errors=102
56500/62423  fetched=55619  errors=102
57000/62423  fetched=56118  errors=102
57500/62423  fetched=56610  errors=102
58000/62423  fetched=57095  errors=102
58500/62423  fetched=57587  errors=102
59000/62423  fetched=58072  errors=102
59500/62423  fetched=58565  errors=107
60000/62423  fetched=59063  errors=107
60500/62423  fetched=59563  errors=107
61000/62423  fetched=60062  errors=107
61500/62423  fetched=60559  errors=107
62000/62423  fetched=61059  errors=107
Done: 61477 fetched, 107 errors


In [9]:
results

[{'movieId': 356,
  'overview': 'A man with a low IQ has accomplished great things in his life and been present during significant historic events—in each case, far exceeding what anyone imagined he could do. But despite all he has achieved, his one true love eludes him.',
  'tagline': "The world will never be the same once you've seen it through the eyes of Forrest Gump.",
  'original_title': 'Forrest Gump',
  'original_language': 'en',
  'release_date': '1994-06-23',
  'release_year': 1994.0,
  'runtime': 142,
  'status': 'Released',
  'vote_average': 8.464,
  'vote_count': 29914,
  'popularity': 29.731,
  'budget': 55000000.0,
  'revenue': 677387716.0,
  'poster_path': '/Cw4hIUIAmSYfK9QfaUW5igp9La.jpg',
  'backdrop_path': '/66Kn4XWhkuPkJxOJyPEx4U2CUfN.jpg',
  'origin_country': 'US',
  'country': 'US',
  'spoken_languages': 'English',
  'collection': nan,
  'production_companies': 'Paramount Pictures|The Steve Tisch Company|Wendy Finerman Productions',
  'director': 'Robert Zemeckis'

## 5. Merge & save

Join the TMDB results back onto the movie list and build the `text` column — a single string combining title, genres, and overview.
This `text` field is the input to the MiniLM sentence encoder in `2_build_embeddings.ipynb`.

Final CSV is saved to `data/movies_clean.csv` with columns:
`movieId, title, genres, tmdbId, overview, release_year, vote_average, popularity, budget, director, country, text`

In [10]:
# Merge TMDB details into the movie list and build a combined text field
# used later by MiniLM to produce description embeddings.
movies_final = movies_popular.merge(tmdb_df, on='movieId', how='left')

# text = title + genres + overview — the input to the sentence encoder
movies_final['text'] = (
    movies_final['title'].fillna('') + ' | ' +
    movies_final['genres'].fillna('') + ' | ' +
    movies_final['overview'].fillna('')
)

movies_final.to_csv(DATA_DIR / 'movies_clean.csv', index=False)

filled_budget   = movies_final['budget'].notna().sum()
filled_director = movies_final['director'].notna().sum()
print(f'Saved {len(movies_final)} movies')
print(f'  budget:   {filled_budget} filled ({filled_budget/len(movies_final)*100:.1f}%)')
print(f'  director: {filled_director} filled ({filled_director/len(movies_final)*100:.1f}%)')


Saved 62423 movies
  budget:   13967 filled (22.4%)
  director: 61248 filled (98.1%)


## 6. Preview

In [11]:
movies_final

,movieId,title,genres,tmdbId,n_ratings,overview,tagline,original_title,original_language,release_date,...,collection,production_companies,director,writer,composer,cinematographer,producer,cast,keywords,text
0,356,Forrest Gump (1994),Comedy|Drama|Romance|War,13.0,81491.0,A man with a low IQ has accomplished great thi...,The world will never be the same once you've s...,Forrest Gump,en,1994-06-23,...,NaN,Paramount Pictures|The Steve Tisch Company|Wen...,Robert Zemeckis,Eric Roth,Alan Silvestri,Don Burgess,Wendy Finerman|Steve Starkey|Steve Tisch,Tom Hanks|Robin Wright|Gary Sinise|Sally Field...,new year's eve|vietnam war|vietnam veteran|men...,Forrest Gump (1994) | Comedy|Drama|Romance|War...
1,318,"Shawshank Redemption, The (1994)",Crime|Drama,278.0,81482.0,Imprisoned in the 1940s for the double murder ...,Fear can hold you prisoner. Hope can set you f...,The Shawshank Redemption,en,1994-09-23,...,NaN,Castle Rock Entertainment,Frank Darabont,Frank Darabont,Thomas Newman,Roger Deakins,Niki Marvin,Tim Robbins|Morgan Freeman|Bob Gunton|William ...,prison|friendship|police brutality|corruption|...,"Shawshank Redemption, The (1994) | Crime|Drama..."
2,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,680.0,79672.0,"A burger-loving hit man, his philosophical par...",You won’t know the facts until you’ve seen the...,Pulp Fiction,en,1994-09-10,...,NaN,Miramax|A Band Apart|Jersey Films,Quentin Tarantino,Roger Avary|Quentin Tarantino|Quentin Tarantino,NaN,Andrzej Sekula,Lawrence Bender,John Travolta|Samuel L. Jackson|Uma Thurman|Br...,drug dealer|drug crime|gangster|dark comedy|st...,Pulp Fiction (1994) | Comedy|Crime|Drama|Thril...
3,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,274.0,74127.0,Clarice Starling is a top student at the FBI's...,To enter the mind of a killer she must challen...,The Silence of the Lambs,en,1991-02-14,...,The Hannibal Lecter Collection,Orion Pictures|Strong Heart|A Luta Continua,Jonathan Demme,Ted Tally,Howard Shore,Tak Fujimoto,Ron Bozman|Edward Saxon|Kenneth Utt,Jodie Foster|Anthony Hopkins|Scott Glenn|Ted L...,prison|based on novel or book|kidnapping|psych...,"Silence of the Lambs, The (1991) | Crime|Horro..."
4,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,603.0,72674.0,"Set in the 22nd century, The Matrix tells the ...",Believe the unbelievable.,The Matrix,en,1999-03-31,...,The Matrix Collection,Village Roadshow Pictures|Groucho II Film Part...,Lana Wachowski,Lana Wachowski|Lilly Wachowski,Don Davis,Bill Pope,Joel Silver,Keanu Reeves|Laurence Fishburne|Carrie-Anne Mo...,man vs machine|martial arts|fortune teller|art...,"Matrix, The (1999) | Action|Sci-Fi|Thriller | ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62418,208411,Eternal Blood (2002),Action|Horror|Thriller,101604.0,NaN,Carmila is introduced by 'M' to a sinister rol...,It's the Essence of Life...and Death.,Sangre eterna,es,2002-10-31,...,NaN,Angel Films Producciones,Jorge Olguín,Jorge Olguín|Carolina García,Gamal Eltit,José Luis Arredondo,Daniel Pantoja,Blanca Lewin|Juan Pablo Ogalde|Patricia López|...,vampire,Eternal Blood (2002) | Action|Horror|Thriller ...
62419,208413,Big Business (1929),Comedy,48649.0,NaN,Stan and Ollie play door-to-door Christmas tre...,The story of a man who turned the other cheek-...,Big Business,en,1929-04-20,...,NaN,Hal Roach Studios,James W. Horne,H.M. Walker,NaN,George Stevens,Hal Roach,Stan Laurel|Oliver Hardy|James Finlayson|Tiny ...,black and white|silent film|short film,Big Business (1929) | Comedy | Stan and Ollie ...
62420,208415,The Student of Prague (1926),Horror,27519.0,NaN,"For Balduin, going out to beer parties with hi...",,Der Student von Prag,de,1926-10-19,...,NaN,Sokal-Film GmbH,Henrik Galeen,Hanns Heinz Ewers|Henrik Galeen,NaN,Erich Nitzschmann,Harry R. Sokal,Conrad Veidt|Elizza La Porta|Fritz Alberti|Agn...,NaN,The Student of Prague (1926) | Horror | For Ba...
62421,208655,The Coldest Game (2019),(no genres listed),585759.0,NaN,"Warsaw